# Notebook 09 — Multi-Outcome LP-IV (Plan Step 6)
## Extension of Saadaoui (2026, JCE)

**Plan step 6.2:** 'Train a model that simultaneously predicts the effect of
a geopolitical shock on multiple outcomes: Oil prices (WTI, Brent), Gasoline,
Sovereign bond yields, Currency volatility, Global equity volatility (VIX).'

We run the same LP-IV (same instrument d2pri, same controls) on every outcome
available in our data. This tests whether the US-China PRI shock propagates
across markets and, if so, in what order (which market responds first).

**Available outcomes:**

| Outcome | Variable | Represents |
|---------|----------|------------|
| WTI crude oil | lwti | Saadaoui baseline |
| Brent crude oil | brent | Alternative oil benchmark |
| Gold (safe haven) | gold | Flight-to-safety response |
| VIX (fear index) | vix | Global risk appetite |
| CNY/USD exchange | cny_usd | Bilateral currency channel |
| Baltic Dry Index | bdi | Trade/shipping activity |
| US 10y yield | gs10 | Safe-haven bond demand |

**Research question:** Does a US-China geopolitical improvement first affect
oil prices, then currencies, then bonds — or is the response simultaneous?
Which markets are most sensitive to bilateral geopolitical shocks?

**Note on endogeneity:** The instrument d2pri is designed for the PRI→WTI
channel. Using it for other outcomes is valid as long as d2pri affects
those outcomes only through PRI (exclusion restriction extends).
This is more credible for oil-linked variables (Brent, BDI) than for
bond yields where other channels exist. We note this caveat per outcome.


In [1]:
from pathlib import Path
import warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.tools import add_constant
from linearmodels.iv import IV2SLS
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)

cwd  = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
FINAL   = ROOT / 'data' / 'final'
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
for d in [RESULTS, FIGURES]: d.mkdir(parents=True, exist_ok=True)

HMAX = 48
print(f'ROOT={ROOT}')


ROOT=C:\Users\HP\Desktop\replication+contribution


In [2]:
df_ext = pd.read_csv(FINAL / 'df_extended.csv', index_col=0, parse_dates=True)
df_ext.index = pd.to_datetime(df_ext.index)

with open(FINAL / 'variable_roles.json') as f:
    roles = json.load(f)

INSTRUMENT = roles['instrument_core'][0]
TREATMENT  = roles['treatment'][0]
OUTCOME_PRIMARY = roles['outcome'][0]   # lwti
CONTROLS = (roles['controls_core'] +
            roles['controls_macro'] +
            roles['controls_geopol'])

# Define outcomes. Each needs:
#   col: column in df_ext
#   label: display name
#   log: already log-transformed? (if not, take log)
#   exclusion_note: caveat on exclusion restriction
OUTCOMES = [
    {'col': 'lwti',    'label': 'WTI crude oil',    'log': True,
     'excl': 'Primary outcome — exclusion well-motivated'},
    {'col': 'brent',   'label': 'Brent crude oil',   'log': False,
     'excl': 'Oil benchmark — same channel as WTI, exclusion credible'},
    {'col': 'gold',    'label': 'Gold (safe haven)', 'log': False,
     'excl': 'Flight-to-safety channel — exclusion plausible'},
    {'col': 'vix',     'label': 'VIX (fear index)',  'log': False,
     'excl': 'Risk appetite channel — exclusion plausible'},
    {'col': 'cny_usd', 'label': 'CNY/USD rate',      'log': False,
     'excl': 'Bilateral currency — direct channel, exclusion credible'},
    {'col': 'bdi',     'label': 'Baltic Dry Index',  'log': False,
     'excl': 'Trade channel — exclusion plausible'},
    {'col': 'gs10',    'label': 'US 10y yield',      'log': False,
     'excl': 'Safe-haven bond demand — exclusion less certain'},
]

# Filter to outcomes that are in the data
OUTCOMES = [o for o in OUTCOMES if o['col'] in df_ext.columns]

print(f'df_extended: n={len(df_ext)} | {df_ext.index.min().date()} to {df_ext.index.max().date()}')
print(f'Available outcomes ({len(OUTCOMES)}):')
for o in OUTCOMES:
    n_obs = df_ext[o['col']].notna().sum()
    print(f'  {o["col"]:12s} | {o["label"]:22s} | n={n_obs} | {o["excl"]}')


df_extended: n=385 | 1990-02-28 to 2022-02-28
Available outcomes (7):
  lwti         | WTI crude oil          | n=385 | Primary outcome — exclusion well-motivated
  brent        | Brent crude oil        | n=385 | Oil benchmark — same channel as WTI, exclusion credible
  gold         | Gold (safe haven)      | n=385 | Flight-to-safety channel — exclusion plausible
  vix          | VIX (fear index)       | n=385 | Risk appetite channel — exclusion plausible
  cny_usd      | CNY/USD rate           | n=385 | Bilateral currency — direct channel, exclusion credible
  bdi          | Baltic Dry Index       | n=385 | Trade channel — exclusion plausible
  gs10         | US 10y yield           | n=385 | Safe-haven bond demand — exclusion less certain


In [3]:
def F_shift(s, h): return s.shift(-h)

def add_lags(df, y_col, shock_col, y_lags=3, shock_lags=2):
    out = df.copy(); lag_cols = []
    for l in range(1, y_lags+1):
        c = f'L{l}_{y_col}'; out[c] = out[y_col].shift(l); lag_cols.append(c)
    for l in range(1, shock_lags+1):
        c = f'L{l}_{shock_col}'; out[c] = out[shock_col].shift(l); lag_cols.append(c)
    return out, lag_cols

def lp_iv_outcome(df, y_col, endog, instr, controls, hmax=HMAX, log_y=False):
    """
    LP-IV for a given outcome y_col.
    Lags of the OUTCOME are always lags of lwti (the primary outcome),
    to keep the same control set across outcomes.
    The forward shift is applied to y_col.
    """
    work = df.copy()
    # Always lag the primary outcome (lwti) as controls
    for l in range(1,4): work[f'L{l}_lwti'] = work[OUTCOME_PRIMARY].shift(l)
    for l in range(1,3): work[f'L{l}_lpri'] = work[endog].shift(l)
    lag_cols = [f'L{l}_lwti' for l in range(1,4)] + [f'L{l}_lpri' for l in range(1,3)]
    exog_cols = lag_cols + controls

    # Outcome: take log if needed
    if log_y:
        outcome_series = work[y_col]
    else:
        outcome_series = np.log(work[y_col].clip(lower=1e-6))

    rows = []
    for h in range(hmax + 1):
        hdf = pd.DataFrame({
            'y_fwd': F_shift(outcome_series, h),
            endog: work[endog], instr: work[instr],
            **{c: work[c] for c in exog_cols},
        }).replace([np.inf,-np.inf], np.nan).dropna()
        if len(hdf) < 40:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n_obs':len(hdf)}); continue
        try:
            fit = IV2SLS(dependent=hdf['y_fwd'],
                         exog=add_constant(hdf[exog_cols], has_constant='add'),
                         endog=hdf[endog], instruments=hdf[instr]
                         ).fit(cov_type='robust', debiased=True)
            rows.append({'h':h,'coef':float(fit.params.get(endog,np.nan)),
                         'se':float(fit.std_errors.get(endog,np.nan)),'n_obs':len(hdf)})
        except:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n_obs':len(hdf)})
    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef']-1.645*irf['se']
    irf['hi90'] = irf['coef']+1.645*irf['se']
    irf['lo95'] = irf['coef']-1.96 *irf['se']
    irf['hi95'] = irf['coef']+1.96 *irf['se']
    return irf

print('Running LP-IV for all outcomes...')
irf_all = {}
for o in OUTCOMES:
    print(f'  {o["label"]}...', end=' ')
    irf_o = lp_iv_outcome(df_ext, o['col'], TREATMENT, INSTRUMENT, CONTROLS,
                          log_y=o['log'])
    irf_all[o['col']] = irf_o
    irf_o.to_csv(RESULTS / f'irf_outcome_{o["col"]}.csv', index=False)
    sig10 = (irf_o['lo90'] > 0).sum() + (irf_o['hi90'] < 0).sum()
    print(f'sig horizons (90% CI excludes 0): {sig10}/{HMAX+1}')

print('All outcomes done.')


Running LP-IV for all outcomes...
  WTI crude oil... sig horizons (90% CI excludes 0): 16/49
  Brent crude oil... sig horizons (90% CI excludes 0): 5/49
  Gold (safe haven)... sig horizons (90% CI excludes 0): 7/49
  VIX (fear index)... sig horizons (90% CI excludes 0): 3/49
  CNY/USD rate... sig horizons (90% CI excludes 0): 7/49
  Baltic Dry Index... sig horizons (90% CI excludes 0): 7/49
  US 10y yield... sig horizons (90% CI excludes 0): 7/49
All outcomes done.


In [4]:
# ── Multi-panel figure ─────────────────────────────────────────────────────────
hs = np.arange(HMAX+1)
n_out = len(OUTCOMES)
ncols = 3
nrows = int(np.ceil(n_out / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5*nrows))
axes = axes.flatten()

COLORS = {
    'lwti':    ('steelblue',   'WTI crude'),
    'brent':   ('firebrick',   'Brent crude'),
    'gold':    ('goldenrod',   'Gold'),
    'vix':     ('purple',      'VIX'),
    'cny_usd': ('teal',        'CNY/USD'),
    'bdi':     ('darkorange',  'BDI'),
    'gs10':    ('darkgreen',   'US 10y yield'),
}

for i, o in enumerate(OUTCOMES):
    ax = axes[i]
    irf = irf_all[o['col']]
    col = COLORS.get(o['col'], ('gray',''))[0]

    ax.plot(hs, irf['coef'], color=col, lw=2.0, label=o['label'])
    ax.fill_between(hs, irf['lo90'], irf['hi90'], color=col, alpha=0.18, label='90% CI')
    ax.fill_between(hs, irf['lo95'], irf['hi95'], color=col, alpha=0.08, label='95% CI')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlim(0,HMAX); ax.set_xticks(np.arange(0,HMAX+1,12))
    ax.set_xlabel('Months after shock')
    ax.set_ylabel(f'IRF of log {o["col"]}')

    sig10 = (irf['lo90']>0).sum() + (irf['hi90']<0).sum()
    ax.set_title(f'{o["label"]}\n'
                 f'Sig at 90%: {sig10}/{HMAX+1} | {o["excl"][:35]}',
                 fontsize=9)
    ax.legend(fontsize=8)

# Hide unused axes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Plan Step 6: Multi-Outcome LP-IV\n'
             'Instrument: Δ²PRI (d2pri) | Treatment: US-China PRI\n'
             'How does a geopolitical improvement propagate across markets?',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES/'Figure_09_multi_outcome.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved: Figure_09_multi_outcome.png')


Saved: Figure_09_multi_outcome.png


In [5]:
# ── Peak response comparison: which market responds first and strongest? ────────
print('MULTI-OUTCOME SUMMARY — Peak responses')
print('=' * 65)
print(f'  {"Outcome":<22}  {"Peak coef":>10}  {"Peak h":>7}  {"Sig 90%":>8}  Exclusion')
print('-' * 80)
peak_rows = []
for o in OUTCOMES:
    irf = irf_all[o['col']]
    valid = irf.dropna(subset=['coef'])
    if valid.empty:
        print(f'  {o["label"]:<22}  no valid estimates')
        continue
    # Peak by absolute magnitude
    peak_idx = valid['coef'].abs().idxmax()
    peak_coef = valid.loc[peak_idx,'coef']
    peak_h    = int(valid.loc[peak_idx,'h'])
    sig10 = (irf['lo90']>0).sum() + (irf['hi90']<0).sum()
    print(f'  {o["label"]:<22}  {peak_coef:>10.4f}  h={peak_h:>5d}  '
          f'{sig10:>5d}/{HMAX+1}  {o["excl"][:30]}')
    peak_rows.append({'outcome':o['label'],'col':o['col'],
                      'peak_coef':peak_coef,'peak_h':peak_h,
                      'sig90':int(sig10)})

pd.DataFrame(peak_rows).to_csv(RESULTS/'multi_outcome_summary.csv', index=False)

print()
print('INTERPRETATION:')
print('  Compare peak_h across outcomes: lower peak_h = faster response.')
print('  The ordering reveals the transmission mechanism:')
print('  If VIX peaks first → risk appetite channel dominates.')
print('  If CNY/USD peaks first → currency channel dominates.')
print('  If WTI peaks first → direct energy supply/demand channel.')
print()
print('CAVEAT:')
print('  Exclusion restriction: d2pri is designed for the PRI→WTI channel.')
print('  Using it for bond yields and VIX assumes no direct d2pri→outcome path.')
print('  Results for gs10 should be interpreted with extra caution.')
print('  Results for brent, gold, cny_usd, bdi are more credible.')
print('Saved: multi_outcome_summary.csv')


MULTI-OUTCOME SUMMARY — Peak responses
  Outcome                  Peak coef   Peak h   Sig 90%  Exclusion
--------------------------------------------------------------------------------
  WTI crude oil               0.2510  h=   34     16/49  Primary outcome — exclusion we
  Brent crude oil             4.0094  h=   19      5/49  Oil benchmark — same channel a
  Gold (safe haven)           2.9524  h=    3      7/49  Flight-to-safety channel — exc
  VIX (fear index)            0.1647  h=   13      3/49  Risk appetite channel — exclus
  CNY/USD rate               -2.5661  h=   27      7/49  Bilateral currency — direct ch
  Baltic Dry Index           -3.9626  h=    2      7/49  Trade channel — exclusion plau
  US 10y yield                0.1626  h=   45      7/49  Safe-haven bond demand — exclu

INTERPRETATION:
  Compare peak_h across outcomes: lower peak_h = faster response.
  The ordering reveals the transmission mechanism:
  If VIX peaks first → risk appetite channel dominates.
  If CN

In [6]:
import pandas as pd
df_raw = pd.read_stata('../data/Saadaoui_2026_JCE.dta')
pri_cols = [c for c in df_raw.columns if 'pri' in c.lower() or 'dlpri' in c.lower()]
print(pri_cols)

['lpri', 'pri', 'pri_jp', 'pri_aus', 'pri_cds', 'pri_fra', 'pri_ger', 'pri_india', 'pri_indo', 'pri_pak', 'pri_rus', 'pri_vn', 'pri_uk', 'lpri_jp', 'dlpri_jp', 'lpri_aus', 'dlpri_aus', 'lpri_cds', 'dlpri_cds', 'lpri_fra', 'dlpri_fra', 'lpri_ger', 'dlpri_ger', 'lpri_india', 'dlpri_india', 'lpri_indo', 'dlpri_indo', 'lpri_pak', 'dlpri_pak', 'lpri_rus', 'dlpri_rus', 'lpri_vn', 'dlpri_vn', 'lpri_uk', 'dlpri_uk', 'dlpri', 'd2pri', 'd2pri_jp']
